# EvoEval: Autonomous Code Agent Evolution Benchmark
### Official Google Colab Interactive Cloud Benchmark Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evoeval/evoeval/blob/main/EvoEval_Colab_Benchmark.ipynb)
&nbsp;
[![Paper Track: NeurIPS 2027](https://img.shields.io/badge/NeurIPS%202027-Datasets%20%26%20Benchmarks-blue.svg)](https://anonymous.4open.science/r/EvoEval-NeurIPS2027/)
&nbsp;
[![License: Apache 2.0](https://img.shields.io/badge/License-Apache_2.0-green.svg)](https://opensource.org/licenses/Apache-2.0)

---

## Overview
**EvoEval** is a longitudinal, multi-cycle evaluation harness designed to systematically quantify **Safety Drift** ($\text{SafetyDrift}$), **Specification Gaming** ($\text{ProxyGap}$), and **Catastrophic Forgetting** ($\text{Retention}$) across recursively self-evolving code generation agents ($G_1$–$G_6$).

This Colab notebook executes directly inside Google Colab's Linux environment (with free NVIDIA T4 GPU / High-RAM CPU support), enabling live inference and benchmarking on actual foundation models via:
1. **OpenRouter API** (`qwen/qwen-2.5-coder-32b-instruct`, `meta-llama/llama-3.2-3b-instruct:free`, `qwen/qwen-2.5-coder-7b-instruct`, etc.)
2. **Local GPU vLLM / Hugging Face pipeline**

### In This Notebook:
- **Step 1**: Environment setup, extracting codebase (`evoeval.zip`), dependency installation, and hardware diagnostics
- **Step 2**: OpenRouter API key configuration and target model specification
- **Step 3**: Live API connectivity check & Mock-vs-Real inference drift analysis
- **Step 4**: End-to-end task execution in verified sandboxes with AST anti-tamper monitoring
- **Step 5**: Multi-cycle agent evolution run across experimental groups ($G_1, G_2, G_4, G_6$)
- **Step 6**: Publication table and figure generation for paper submissions
- **Step 7**: Packaging and downloading experimental results and execution trajectories


## Step 1: Environment Setup & Codebase Extraction
This cell checks your GPU, locates/extracts the `evoeval` benchmark codebase (from `evoeval.zip` or existing directory), and installs the dependencies.

In [ ]:
# 1. Hardware Diagnostics
!nvidia-smi

# 2. Locate or Extract EvoEval Repository
import os, sys, zipfile
from pathlib import Path

repo_found = False
repo_dir = None

# Check if already in an extracted evoeval directory
for candidate in [Path.cwd(), Path("/content"), Path("/content/evoeval"), Path("/content/EvoEval")]:
    if (candidate / "evaeval").is_dir():
        repo_dir = candidate
        repo_found = True
        break

if not repo_found:
    # Check if evoeval.zip is in /content or cwd
    for zc in [Path("/content/evoeval.zip"), Path.cwd() / "evoeval.zip"]:
        if zc.exists():
            print(f">>> Found {zc}! Extracting to /content/evoeval...")
            with zipfile.ZipFile(zc, 'r') as zf:
                zf.extractall("/content/evoeval")
            repo_dir = Path("/content/evoeval")
            repo_found = True
            break

if not repo_found:
    # Prompt user to upload evoeval.zip (4.5 MB)
    print(">>> Repository not found. Please upload 'evoeval.zip' (4.5 MB):")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".zip"):
                with zipfile.ZipFile(fn, 'r') as zf:
                    zf.extractall("/content/evoeval")
                repo_dir = Path("/content/evoeval")
                repo_found = True
                break
    except Exception as e:
        print(">>> File upload widget error:", e)

if repo_found and repo_dir:
    os.chdir(repo_dir)
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
    print(f"\n>>> Active Directory: {os.getcwd()}")
    print(">>> Installing EvoEval in editable mode...")
    !pip install -q --upgrade pip
    !pip install -q -e .
    print("\n\u2705 EvoEval successfully installed and ready!")
else:
    print("\u274c ERROR: Please upload 'evoeval.zip' using the Colab Files panel on the left and re-run this cell.")

## Step 2: Configure OpenRouter API Key & Target Model
Provide your OpenRouter API key below. You can either use Google Colab Secrets (recommended) or pass the key directly in the cell.

In [ ]:
import os

# Retrieve API key: checks Colab Secrets, then environment, then default
try:
    from google.colab import userdata
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception:
    api_key = os.environ.get('OPENROUTER_API_KEY')

if not api_key:
    # Default key provided for testing
    api_key = "your-openrouter-api-key-here"

# Target Foundation Model on OpenRouter
# Options:
# - 'qwen/qwen-2.5-coder-32b-instruct' (State-of-the-art coding, recommended)
# - 'qwen/qwen-2.5-coder-7b-instruct' (Canonical paper baseline)
# - 'meta-llama/llama-3.2-3b-instruct:free' (Free tier)
# - 'meta-llama/llama-3.1-8b-instruct' (Cross-family replication)
TARGET_MODEL = "qwen/qwen-2.5-coder-32b-instruct"

os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
os.environ["OPENAI_API_KEY"] = api_key
os.environ["VLLM_API_BASE"] = "https://openrouter.ai/api/v1"
os.environ["VLLM_API_KEY"] = api_key
os.environ["VLLM_MODEL_NAME"] = TARGET_MODEL

print("Environment configured:")
print(f"- API Base:     {os.environ['OPENAI_API_BASE']}")
print(f"- Model:        {TARGET_MODEL}")
print(f"- Key Redacted: {api_key[:12]}...{api_key[-6:]}")

## Step 3: Validate API Key & Query Model Status
Verify connection to OpenRouter, check key quotas, and ensure endpoint responds.

In [ ]:
import httpx
import json

headers = {
    "Authorization": f"Bearer {api_key}",
    "HTTP-Referer": "https://github.com/evoeval/evoeval",
    "X-Title": "EvoEval Colab Runner",
}

with httpx.Client(timeout=15.0) as client:
    auth_res = client.get("https://openrouter.ai/api/v1/auth/key", headers=headers)
    if auth_res.status_code == 200:
        auth_data = auth_res.json().get('data', {})
        print("\u2705 API Key Authentication Successful!")
        print(f"- Label:        {auth_data.get('label')}")
        print(f"- Usage:        ${auth_data.get('usage', 0):.4f}")
        print(f"- Limit:        {auth_data.get('limit') or 'Unlimited / Pay-as-you-go'}")
        print(f"- Is Free Tier: {auth_data.get('is_free_tier')}")
    else:
        print(f"\u274c Authentication Error ({auth_res.status_code}): {auth_res.text}")

## Step 4: Run Live LLM Smoke Test & Measure Mock-vs-Real Drift
Executes `scripts/run_vllm_smoke_test.py` against the live model to test real prompt formatting, code extraction, and sandbox execution on initial tasks (`task_001`, `task_006`).

In [ ]:
import sys, os
from pathlib import Path

# Ensure repo root is active and in sys.path
for p in [Path.cwd(), Path("/content"), Path("/content/evoeval"), Path("/content/EvoEval")]:
    if (p / "evaeval").is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        if os.getcwd() != str(p):
            os.chdir(p)
        break

!python scripts/run_vllm_smoke_test.py \
    --api-base https://openrouter.ai/api/v1 \
    --api-key $api_key \
    --model "$TARGET_MODEL" \
    --tasks task_001,task_006 \
    --output-dir experiments/runs/colab_smoke_test

## Step 5: Execute Multi-Cycle Agent Evolution Experiment
Run a live evaluation comparing agent archetypes:
- **G1 (Zero-Shot Baseline)**: Static zero-shot prompt, no state mutation.
- **G2 (Prompt-Rewriting)**: Dynamically rewrites internal instructions upon failure.
- **G4 (Reflective Memory)**: In-context memory accumulation without verification gates (causes safety drift).
- **G6 (Regression-Guarded)**: Evolutionary search guarded by canary suites and automated rollback.

*(You can adjust `TASKS_TO_RUN`, `CYCLES`, and `GROUPS` below to fit your time and quota).* 

In [ ]:
import sys, os
from pathlib import Path
import time

# Ensure repo root is active and in sys.path
for p in [Path.cwd(), Path("/content"), Path("/content/evoeval"), Path("/content/EvoEval")]:
    if (p / "evaeval").is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        if os.getcwd() != str(p):
            os.chdir(p)
        break

from evaeval.config.models import ExperimentConfig, ModelConfig, SandboxConfig, TasksSplitConfig
from evaeval.environment.task_loader import TaskLoader
from evaeval.llm.client import OpenAICompatibleClient
from evaeval.runner.orchestrator import ExperimentOrchestrator
from evaeval.runner.analysis import AnalysisPipeline

# Experiment Configuration
TASKS_TO_RUN = ["task_001", "task_002", "task_003", "task_006"]  # 4 tasks (standard + drift probe)
GROUPS = ["G1", "G2", "G4", "G6"]
CYCLES = 2  # Longitudinal evolution across 2 cycles
SEEDS = [42]

model_cfg = ModelConfig(
    name=TARGET_MODEL,
    api_base="https://openrouter.ai/api/v1",
    api_key=api_key,
    temperature=0.2,
    max_tokens=2048,
)

exp_config = ExperimentConfig(
    name="colab_live_experiment",
    description="Live foundation model evolutionary evaluation on Google Colab",
    model=model_cfg,
    groups=GROUPS,
    cycles=CYCLES,
    seeds=SEEDS,
    tasks=TasksSplitConfig(train=TASKS_TO_RUN, test=TASKS_TO_RUN),
    max_tasks_per_cycle=len(TASKS_TO_RUN),
    sandbox=SandboxConfig(timeout_sec=45),
)

loader = TaskLoader(Path("tasks/tasks_index.json"))
client = OpenAICompatibleClient(model_cfg, allow_fallback=True)

runs_root = Path("experiments/runs")
orchestrator = ExperimentOrchestrator(
    config=exp_config,
    task_loader=loader,
    runs_dir=runs_root,
    llm_client=client,
    max_workers=2,
)

print(f">>> Launching live evolution benchmark across {len(GROUPS)} groups, {CYCLES} cycles...")
t_start = time.time()
run_dir = orchestrator.run_experiment(run_id="colab_live_experiment")
t_elapsed = time.time() - t_start
print(f"\u2705 Experiment finished in {t_elapsed:.1f}s. Results directory: {run_dir}")

## Step 6: Compute Scientific Metrics & Generate Paper Visualizations
Computes $\Delta P$, $\text{SafetyDrift}$, $\text{ProxyGap}$, and $\text{Retention}$, then exports LaTeX tables and figures.

In [ ]:
import sys, os
from pathlib import Path

for p in [Path.cwd(), Path("/content"), Path("/content/evoeval"), Path("/content/EvoEval")]:
    if (p / "evaeval").is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        if os.getcwd() != str(p):
            os.chdir(p)
        break

from IPython.display import Image, display
from evaeval.runner.analysis import AnalysisPipeline

pipeline = AnalysisPipeline(run_dir=run_dir, force_recompute=True)
figures = pipeline.generate_all_figures(output_dir=run_dir / "figures")
tables = pipeline.export_latex_tables(target_dir=run_dir / "tables")

print("Generated Figures:")
for fig in figures:
    print(f" - {fig.name}")

print("\nGenerated LaTeX Tables:")
for name, tbl in tables.items():
    print(f" - {name}: {tbl.name}")

# Display primary Safety Drift figure
drift_fig = run_dir / "figures" / "safety_drift.png"
if drift_fig.exists():
    display(Image(filename=str(drift_fig)))
else:
    print("Figure generated at:", figures[0] if figures else "N/A")

## Step 7: Package & Download Results
Create an archive containing all execution trajectories (`trajectory.jsonl`), cycle metrics (`cycle_metrics.json`), LaTeX tables, and publication figures, and download it to your local machine.

In [ ]:
from google.colab import files
import shutil

archive_path = "evoeval_colab_results.tar.gz"
!tar -czvf {archive_path} -C experiments/runs colab_live_experiment

print("\u2705 Results archive created! Downloading now...")
files.download(archive_path)